# 05 — Fine-tune DistilBERT (Colab GPU)

**Where to run:** Google Colab with a T4 GPU (free tier is fine).

**Setup before running:**
1. Upload `data/processed/train.parquet`, `val.parquet`, `test.parquet` to a folder in your Drive (e.g. `MyDrive/nlp_project/data/`).
2. Mount Drive in the cell below.
3. After training, download the test predictions parquet back into your local `results/predictions/` folder.

Model: `distilbert-base-uncased` — general-purpose pre-trained transformer.

In [ ]:
!pip install -q transformers datasets evaluate accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/nlp_project/data'
OUT_DIR = '/content/drive/MyDrive/nlp_project/results'
import os; os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import time, numpy as np, pandas as pd, torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding,
)
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score, precision_recall_fscore_support,
)

print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '—')
torch.manual_seed(42); np.random.seed(42)

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
MODEL_KEY = 'distilbert'

train = pd.read_parquet(f'{DATA_DIR}/train.parquet')
val   = pd.read_parquet(f'{DATA_DIR}/val.parquet')
test  = pd.read_parquet(f'{DATA_DIR}/test.parquet')

label_to_int = {'activist': 0, 'sceptic': 1}
int_to_label = {0: 'activist', 1: 'sceptic'}

for df in (train, val, test):
    df['label'] = df['stance'].map(label_to_int)

print(f'train: {len(train):,}, val: {len(val):,}, test: {len(test):,}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 96

def tok(batch):
    return tokenizer(batch['clean_text'], truncation=True, max_length=MAX_LEN, padding=False)

def to_ds(df):
    ds = Dataset.from_pandas(df[['clean_text', 'label']].astype({'clean_text': str}), preserve_index=False)
    return ds.map(tok, batched=True)

train_ds, val_ds, test_ds = to_ds(train), to_ds(val), to_ds(test)
collator = DataCollatorWithPadding(tokenizer)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=int_to_label, label2id=label_to_int,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
    }

args = TrainingArguments(
    output_dir=f'/content/out_{MODEL_KEY}',
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    save_total_limit=1,
    weight_decay=0.01,
    warmup_ratio=0.06,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    seed=42,
    report_to='none',
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    tokenizer=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics,
)
t0 = time.time()
trainer.train()
print(f'train time: {(time.time() - t0)/60:.1f} min')

In [ ]:
preds_out = trainer.predict(test_ds)
logits = preds_out.predictions
y_pred_int = logits.argmax(axis=-1)
y_pred_str = np.array([int_to_label[i] for i in y_pred_int])
y_test_str = test['stance'].values
y_proba_sceptic = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

p, r, f, _ = precision_recall_fscore_support(y_test_str, y_pred_str, labels=['activist', 'sceptic'], zero_division=0)
metrics = {
    'model': MODEL_KEY,
    'accuracy': float(accuracy_score(y_test_str, y_pred_str)),
    'f1_macro': float(f1_score(y_test_str, y_pred_str, average='macro')),
    'f1_weighted': float(f1_score(y_test_str, y_pred_str, average='weighted')),
    'precision_activist': float(p[0]), 'recall_activist': float(r[0]), 'f1_activist': float(f[0]),
    'precision_sceptic': float(p[1]), 'recall_sceptic': float(r[1]), 'f1_sceptic': float(f[1]),
    'roc_auc': float(roc_auc_score((y_test_str == 'sceptic').astype(int), y_proba_sceptic)),
}
print(classification_report(y_test_str, y_pred_str, digits=4))
print(metrics)

In [ ]:
# Save predictions + metrics back to Drive — download these to local results/ for notebook 07
pd.DataFrame({
    'y_true': y_test_str, 'y_pred': y_pred_str, 'proba_sceptic': y_proba_sceptic,
}).to_parquet(f'{OUT_DIR}/{MODEL_KEY}.parquet', index=False)
pd.DataFrame([metrics]).to_csv(f'{OUT_DIR}/metrics_{MODEL_KEY}.csv', index=False)
trainer.save_model(f'{OUT_DIR}/model_{MODEL_KEY}')
print('Saved predictions, metrics, and model to', OUT_DIR)